# Build final metadata parquet (with `split`)

Merges all caption levels and the split column into a single
`metadata_<size>.parquet` (+ CSV) for one patch size.

**Inputs (all in `BASEDIR`):**

- `captions_<size>_level1.parquet`, `level2`, `level3`, `bigearthnet`, `custom`
- `split_summary.parquet`  — primary 70/15/15 split (provides `split`)

**Output:**

`metadata_<size>.parquet` + `.csv` with columns:

```
BaseFolder, BaseFilename, CLC_codes,
caption1, caption2, caption3, caption4, caption5,
split
```

The merge keys on `BaseFilename`. The script asserts all rows have non-null
`split` after merging, so missing entries fail loudly.


In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Merge caption parquet files with primary split table.

Auto-detects the caption column name (case-insensitive) in each caption file,
then merges all caption levels + split into metadata_<size>.parquet/.csv.
"""

import os
import pandas as pd

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
BASEDIR    = "/home/ubuntu/SENSERO/GeoTiff/Patch_336"
PATCH_SIZE = 336

# Caption files (suffix -> output column name)
CAP_FILES = {
    "caption1": f"captions_{PATCH_SIZE}_level1.parquet",
    "caption2": f"captions_{PATCH_SIZE}_level2.parquet",
    "caption3": f"captions_{PATCH_SIZE}_level3.parquet",
    "caption4": f"captions_{PATCH_SIZE}_bigearthnet.parquet",
    "caption5": f"captions_{PATCH_SIZE}_custom.parquet",
}

# Split file (must live in BASEDIR for this patch size)
SPLIT_FILE = "split_summary.parquet"   # primary 70/15/15 -> provides `split`

# ------------------------------------------------------------------
# HELPER: Load one caption parquet, auto-detecting the caption column
# ------------------------------------------------------------------
def load_caption(path, caption_colname):
    df = pd.read_parquet(path)
    cols = list(df.columns)

    # Find the caption column (first column whose name contains 'caption')
    caption_candidates = [c for c in cols if "caption" in c.lower()]
    if not caption_candidates:
        raise KeyError(f"No caption-like column found in {path}")
    caption_col = caption_candidates[0]

    expected = ["BaseFolder", "BaseFilename", "CLC_codes"]
    missing = [c for c in expected if c not in cols]
    if missing:
        raise KeyError(f"Missing expected columns {missing} in {path}")

    out = df[["BaseFolder", "BaseFilename", "CLC_codes", caption_col]].copy()
    out = out.rename(columns={caption_col: caption_colname})
    return out

# ------------------------------------------------------------------
# LOAD CAPTIONS
# ------------------------------------------------------------------
captions = {}
for cname, fname in CAP_FILES.items():
    fpath = os.path.join(BASEDIR, fname)
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"Missing caption file: {fpath}")
    captions[cname] = load_caption(fpath, cname)

# ------------------------------------------------------------------
# LOAD SPLIT
# ------------------------------------------------------------------
split_path = os.path.join(BASEDIR, SPLIT_FILE)
if not os.path.exists(split_path):
    raise FileNotFoundError(f"Missing split file: {split_path}")

split_df = pd.read_parquet(split_path)
required = {"BaseFilename", "split"}
missing = required - set(split_df.columns)
if missing:
    raise KeyError(f"{SPLIT_FILE} is missing columns: {missing}")
split_df = split_df[["BaseFilename", "split"]].copy()

# ------------------------------------------------------------------
# MERGE
# ------------------------------------------------------------------
merged = captions["caption1"].copy()
for cname in ["caption2", "caption3", "caption4", "caption5"]:
    merged = merged.merge(
        captions[cname][["BaseFilename", cname]],
        on="BaseFilename",
        how="left",
    )

merged = merged.merge(split_df, on="BaseFilename", how="left")

# ------------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------------
n_missing_split = merged["split"].isna().sum()
if n_missing_split:
    raise ValueError(
        f"After merge: {n_missing_split} rows have null `split`. "
        f"Check that {SPLIT_FILE} covers every BaseFilename."
    )

# Column order: identifiers, captions, then split at the end
col_order = (
    ["BaseFolder", "BaseFilename", "CLC_codes"]
    + list(CAP_FILES.keys())
    + ["split"]
)
merged = merged[col_order]

# ------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------
out_parquet = os.path.join(BASEDIR, f"metadata_{PATCH_SIZE}.parquet")
out_csv     = os.path.join(BASEDIR, f"metadata_{PATCH_SIZE}.csv")

# Atomic writes so a crash mid-save can't corrupt the file
tmp_parquet = out_parquet + ".tmp"
tmp_csv     = out_csv + ".tmp"
merged.to_parquet(tmp_parquet, index=False)
merged.to_csv(tmp_csv, index=False, encoding="utf-8")
os.replace(tmp_parquet, out_parquet)
os.replace(tmp_csv,     out_csv)

# ------------------------------------------------------------------
# REPORT
# ------------------------------------------------------------------
print(f"[OK] Merged metadata saved:")
print(f"  Parquet: {out_parquet}")
print(f"  CSV    : {out_csv}")
print(f"Rows: {len(merged):,}")
print(f"Columns: {list(merged.columns)}")

print("\nsplit (70/15/15) distribution:")
print(merged["split"].value_counts())

print("\nPreview:")
print(merged.head(5))


[OK] Merged metadata saved:
  Parquet: /home/ubuntu/SENSERO/GeoTiff/Patch_336/metadata_336.parquet
  CSV    : /home/ubuntu/SENSERO/GeoTiff/Patch_336/metadata_336.csv
Rows: 10,000
Columns: ['BaseFolder', 'BaseFilename', 'CLC_codes', 'caption1', 'caption2', 'caption3', 'caption4', 'caption5', 'split']

split (70/15/15) distribution:
split
train    7000
val      1500
test     1500
Name: count, dtype: int64

Preview:
                                     BaseFolder  \
0  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM   
1  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM   
2  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM   
3  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM   
4  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM   

                                        BaseFilename  \
0  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1...   
1  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1...   
2  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1...   
3  S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1...